# Section G — Governance and Security

Privileges, entitlement-driven column masks, and row-level filters for the FreshKart Lakehouse.

In [0]:
catalog = "fk_ce_2026_07_1378"

### G1 — Privileges

In [0]:
# G1: Grant privileges to fk_analysts and fk_engineers groups

# fk_analysts: USE CATALOG; USE SCHEMA + SELECT on gold only
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `fk_analysts`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.gold TO `fk_analysts`")
spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.gold TO `fk_analysts`")

# fk_engineers: USE CATALOG; USE SCHEMA + SELECT + MODIFY on bronze/silver/gold/ops
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `fk_engineers`")
for schema in ["bronze", "silver", "gold", "ops"]:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `fk_engineers`")
    spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.{schema} TO `fk_engineers`")
    spark.sql(f"GRANT MODIFY ON SCHEMA {catalog}.{schema} TO `fk_engineers`")

# fk_engineers: READ VOLUME + WRITE VOLUME on landing.raw
spark.sql(f"GRANT READ VOLUME ON VOLUME {catalog}.landing.raw TO `fk_engineers`")
spark.sql(f"GRANT WRITE VOLUME ON VOLUME {catalog}.landing.raw TO `fk_engineers`")

# Verify
print("=== fk_analysts ===")
spark.sql(f"SHOW GRANTS `fk_analysts` ON CATALOG {catalog}").show(truncate=False)
print("=== fk_engineers ===")
spark.sql(f"SHOW GRANTS `fk_engineers` ON CATALOG {catalog}").show(truncate=False)

### G2 — Entitlement Table

In [0]:
# G2: Create ops.user_entitlements table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.user_entitlements (
    user_email STRING,
    allowed_city STRING,
    can_view_pii BOOLEAN
)
""")
print(f"Rows: {spark.table(f'{catalog}.ops.user_entitlements').count()}")
spark.sql(f"DESCRIBE {catalog}.ops.user_entitlements").show()

### G3 — Column Masks

In [0]:
# G3: Column masks on gold.dim_customer for email and phone
# email → r***@gmail.com (first char before @, then ***@, then domain)
# phone → XXXXXX3366 (XXXXXX + last 4 digits)

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.ops.mask_email(val STRING)
RETURNS STRING
RETURN CASE
    WHEN val IS NULL THEN NULL
    WHEN is_member('fk_admins') THEN val
    WHEN EXISTS (
        SELECT 1 FROM {catalog}.ops.user_entitlements
        WHERE user_email = current_user() AND can_view_pii = TRUE
    ) THEN val
    ELSE CONCAT(SUBSTRING(SPLIT(val, '@')[0], 1, 1), '***@', SPLIT(val, '@')[1])
END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.ops.mask_phone(val STRING)
RETURNS STRING
RETURN CASE
    WHEN val IS NULL THEN NULL
    WHEN is_member('fk_admins') THEN val
    WHEN EXISTS (
        SELECT 1 FROM {catalog}.ops.user_entitlements
        WHERE user_email = current_user() AND can_view_pii = TRUE
    ) THEN val
    ELSE CONCAT('XXXXXX', RIGHT(val, 4))
END
""")

spark.sql(f"ALTER TABLE {catalog}.gold.dim_customer ALTER COLUMN email SET MASK {catalog}.ops.mask_email")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_customer ALTER COLUMN phone SET MASK {catalog}.ops.mask_phone")

print("=== Mask functions created and applied ===")
spark.sql(f"DESCRIBE {catalog}.gold.dim_customer").show(truncate=False)

### G4 — Row Filter

In [0]:
# G4: Row filter on gold.agg_daily_store_sales
# fk_admins → all rows; allowed_city='ALL' → all rows;
# allowed_city=<city> → only that city; no entitlement → no rows

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.ops.city_filter(city_val STRING)
RETURNS BOOLEAN
RETURN CASE
    WHEN is_member('fk_admins') THEN TRUE
    WHEN EXISTS (
        SELECT 1 FROM {catalog}.ops.user_entitlements
        WHERE user_email = current_user()
        AND (allowed_city = 'ALL' OR allowed_city = city_val)
    ) THEN TRUE
    ELSE FALSE
END
""")

spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_store_sales SET ROW FILTER {catalog}.ops.city_filter ON (city)")

print("=== Row filter applied ===")
spark.sql(f"DESCRIBE {catalog}.gold.agg_daily_store_sales").show(truncate=False)

### G5 — Evidence and Persistence

**NOTE**: Run this cell AFTER Run 2 (Batch 2 product load).

In [0]:
# G5: Evidence and persistence — run AFTER Run 2
current_user_email = spark.sql("SELECT current_user()").collect()[0][0]

# Step 1: Set entitlement to Jaipur, can_view_pii = FALSE
spark.sql(f"""
MERGE INTO {catalog}.ops.user_entitlements t
USING (SELECT '{current_user_email}' AS user_email, 'Jaipur' AS allowed_city, FALSE AS can_view_pii) s
ON t.user_email = s.user_email
WHEN MATCHED THEN UPDATE SET allowed_city = s.allowed_city, can_view_pii = s.can_view_pii
WHEN NOT MATCHED THEN INSERT (user_email, allowed_city, can_view_pii) VALUES (s.user_email, s.allowed_city, s.can_view_pii)
""")

# Step 2: Query protected tables as yourself
row_count = spark.table(f"{catalog}.gold.agg_daily_store_sales").count()
email_val = spark.sql(f"SELECT email FROM {catalog}.gold.dim_customer WHERE customer_id = 'C0001'").collect()[0][0]
phone_val = spark.sql(f"SELECT phone FROM {catalog}.gold.dim_customer WHERE customer_id = 'C0001'").collect()[0][0]

# Step 3: Record evidence (exactly 3 rows)
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.governance_evidence (
    evidence_name STRING,
    observed_value STRING
)
""")
spark.sql(f"""
INSERT INTO {catalog}.ops.governance_evidence VALUES
('ROW_FILTER_ROW_COUNT_JAIPUR', CAST({row_count} AS STRING)),
('MASKED_EMAIL_C0001', '{email_val}'),
('MASKED_PHONE_C0001', '{phone_val}')
""")

# Step 4: Change entitlement to ALL, can_view_pii = FALSE
spark.sql(f"UPDATE {catalog}.ops.user_entitlements SET allowed_city = 'ALL' WHERE user_email = '{current_user_email}'")

# Verify
print("=== Governance evidence ===")
display(spark.table(f"{catalog}.ops.governance_evidence"))
print("=== Final entitlement (should be 1 row, ALL/FALSE) ===")
display(spark.table(f"{catalog}.ops.user_entitlements"))
print(f"Row count with ALL entitlement: {spark.table(f'{catalog}.gold.agg_daily_store_sales').count()}")